In [1]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

In [2]:
global_seed = 0
global_rng = np.random.default_rng(global_seed)

In [3]:
mnist = fetch_openml('mnist_784', parser='auto')
X = mnist.data
y = mnist.target.astype(int).to_numpy()
C = np.unique(y).shape[0]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=global_seed, stratify=y
)

In [10]:
pre_pipe = make_pipeline(
    MinMaxScaler(),
    PCA(n_components=0.93, svd_solver="full", random_state=0),
)

X_pca = pre_pipe.fit_transform(X_train)
X_aug = np.hstack([X_pca, np.ones((X_pca.shape[0], 1))])

## Single, consistent pipeline

One pipeline handles scaling, PCA, and classification end to end, so the same
fitted transforms are reused for CV, grid search, and the test set. This
replaces the two disconnected pipelines (`pipe` and `train_pipeline`) and the
unused manual bias column (`X_aug`) from the original notebook.

In [13]:
full_pipe = make_pipeline(
    pre_pipe,
    StandardScaler(),
    LogisticRegression(
        # C=1.0,
        penalty=None,
        solver='lbfgs',
        max_iter=1000,
        random_state=global_seed
    )
)

## Cross-validation with multiclass-aware metrics

This is 10-class classification, so `recall`, `f1`, and `roc_auc` must use
multiclass-compatible scorers (`_macro` / `_ovr`) rather than the binary
defaults used in the original.

In [14]:
kf = KFold(n_splits=5, shuffle=True, random_state=global_seed)
scoring = ['accuracy', 'recall_macro', 'f1_macro', 'roc_auc_ovr']
cv_results = cross_validate(
    full_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=False
)
for metric in scoring:
    key = f'test_{metric}'
    print(f"{metric:12s}: {cv_results[key].mean():.4f} +/- {cv_results[key].std():.4f}")

KeyboardInterrupt: 

In [ ]:
# Hyperparameter tuning
param_grid = {'logisticregression__C': [0.01, 0.1, 1.0, 10.0]}
grid = GridSearchCV(full_pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)
print(f"Best inv regularization param C: {grid.best_params_['logisticregression__C']}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")

# Evaluate on test set (full_pipeline includes all preprocessing, so raw X_test goes in directly)
best = grid.best_estimator_
y_pred = best.predict(X_test)
y_proba = best.predict_proba(X_test)

print("\nTest set results:")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba, multi_class='ovr'):.4f}")